# LumenY 11 - Dual Binary ATR-Reach Classifier (4H horizon)

**Features:** combined set (features_6 + features_9 + features/)  
**Target:** two binary models:
- `model_long`:  P(max(high[t+1:t+4]) >= close[t] + 0.75*ATR14)
- `model_short`: P(min(low[t+1:t+4])  <= close[t] - 0.75*ATR14)

**Signal:** long when P(long)>thresh, short when P(short)>thresh  
**Train cutoff:** 2024-06-30  
**Models saved to:** `backend/models_11/`

In [52]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import log_loss, roc_auc_score

DATA_DIR   = Path('../backend/data/features_combined')
PRICE_DIR  = Path('../backend/data/processed')
MODELS_DIR = Path('../backend/models_11')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MAJORS    = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'USDCAD', 'AUDUSD', 'NZDUSD']
TRAIN_END = '2024-06-30'
ATR_MULT  = 0.75   # price must reach 0.75x ATR14 in direction within 4H

print('Ready.')
print(f'ATR multiplier: {ATR_MULT}x ATR14')
print(f'Train cutoff:   {TRAIN_END}')


Ready.
ATR multiplier: 0.75x ATR14
Train cutoff:   2024-06-30


## 1. Load, Clean & Build Labels â€” Per Pair, Float32

In [53]:
COLS_DROP_SUFFIX = '_f9'
COLS_DROP_SUBSTR = '_1W'
LOOKAHEAD_COLS   = {
    'mfe_long_pips', 'mfe_short_pips',
    'trail_long_bars', 'trail_short_bars',
    'trail_stop_pips', 'mfe_atr_24',
}

dfs_train = []
dfs_test  = []

for pair in MAJORS:
    print(f'  {pair}...', flush=True)

    df = pd.read_parquet(DATA_DIR / f'{pair}_combined.parquet')
    drop = [c for c in df.columns if (
        c.endswith(COLS_DROP_SUFFIX) or COLS_DROP_SUBSTR in c or
        c == 'pair_id' or c in LOOKAHEAD_COLS
    )]
    df.drop(columns=drop, errors='ignore', inplace=True)

    price_df  = pd.read_parquet(PRICE_DIR / f'{pair}_1H.parquet')
    close = price_df['close']
    high  = price_df['high']
    low   = price_df['low']

    tr  = pd.concat([high - low,
                     (high - close.shift()).abs(),
                     (low  - close.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()

    close_arr = close.reindex(df.index).values
    high_arr  = high.reindex(df.index).values
    low_arr   = low.reindex(df.index).values
    atr_arr   = atr.reindex(df.index).values
    n = len(df)

    label_long  = np.full(n, np.nan)
    label_short = np.full(n, np.nan)

    for i in range(n - 4):
        if np.isnan(atr_arr[i]) or np.isnan(close_arr[i]):
            continue
        thresh       = ATR_MULT * atr_arr[i]
        long_target  = close_arr[i] + thresh
        short_target = close_arr[i] - thresh
        long_hit = short_hit = 0
        for h in range(1, 5):
            j = i + h
            if high_arr[j] >= long_target:  long_hit  = 1
            if low_arr[j]  <= short_target: short_hit = 1
        label_long[i]  = long_hit
        label_short[i] = short_hit

    df['label_long']  = label_long
    df['label_short'] = label_short
    df['pair'] = pair

    feat_cols = [c for c in df.columns if c not in ('label_long', 'label_short', 'pair')]
    df[feat_cols] = df[feat_cols].astype(np.float32)

    dfs_train.append(df[df.index <= TRAIN_END])
    dfs_test.append(df[df.index >  TRAIN_END])
    del df; gc.collect()

print('Concatenating...')
df_train = pd.concat(dfs_train).sort_index(); del dfs_train; gc.collect()
df_test  = pd.concat(dfs_test).sort_index();  del dfs_test;  gc.collect()

feature_cols = [c for c in df_train.columns if c not in ('label_long', 'label_short', 'pair')]

valid = df_train['label_long'].notna()
ll = df_train.loc[valid, 'label_long']
ls = df_train.loc[valid, 'label_short']
print(f'Features:      {len(feature_cols)}')
print(f'Train rows:    {len(df_train):,}')
print(f'Test rows:     {len(df_test):,}')
print(f'Long hit rate: {ll.mean():.1%}')
print(f'Short hit rate:{ls.mean():.1%}')
print(f'RAM train: {df_train.memory_usage(deep=False).sum()/1024**2:.0f} MB')


  EURUSD...
  GBPUSD...
  USDJPY...
  USDCHF...
  USDCAD...
  AUDUSD...
  NZDUSD...
Concatenating...
Features:      625
Train rows:    638,893
Test rows:     65,022
Long hit rate: 48.8%
Short hit rate:50.3%
RAM train: 1546 MB


## 2. Prepare Arrays

In [54]:
valid_train   = df_train['label_long'].notna()
X_clean       = df_train.loc[valid_train, feature_cols].ffill().fillna(0)
y_long_train  = df_train.loc[valid_train, 'label_long'].astype(np.int8)
y_short_train = df_train.loc[valid_train, 'label_short'].astype(np.int8)
del df_train; gc.collect()

valid_test    = df_test['label_long'].notna()
X_test        = df_test.loc[valid_test, feature_cols].ffill().fillna(0)
y_long_test   = df_test.loc[valid_test, 'label_long'].astype(np.int8)
y_short_test  = df_test.loc[valid_test, 'label_short'].astype(np.int8)
pair_test     = df_test.loc[valid_test, 'pair']
del df_test; gc.collect()

print(f'Train: {len(X_clean):,} rows  long_hit={y_long_train.mean():.1%}  short_hit={y_short_train.mean():.1%}')
print(f'Test:  {len(X_test):,} rows   long_hit={y_long_test.mean():.1%}   short_hit={y_short_test.mean():.1%}')
print(f'RAM:   {X_clean.memory_usage(deep=False).sum()/1024**2:.0f} MB')


Train: 622,488 rows  long_hit=48.8%  short_hit=50.3%
Test:  63,680 rows   long_hit=44.6%   short_hit=53.5%
RAM:   1489 MB


## 3. Walk-Forward Splits

In [55]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    splits = []
    test_size = int(n * test_ratio)
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

splits = walk_forward_splits(len(X_clean))
print(f'Walk-forward splits: {len(splits)}')
for i, (tr, te) in enumerate(splits):
    print(f'  Fold {i+1}: train -> {X_clean.index[tr[-1]].date()} ({len(tr):,}) | '
          f'test {X_clean.index[te[0]].date()} -> {X_clean.index[te[-1]].date()} ({len(te):,})')

Walk-forward splits: 5
  Fold 1: train -> 2017-01-23 (311,244) | test 2017-01-23 -> 2018-07-10 (62,248)
  Fold 2: train -> 2018-07-10 (373,492) | test 2018-07-10 -> 2020-01-07 (62,248)
  Fold 3: train -> 2020-01-07 (435,740) | test 2020-01-07 -> 2021-08-02 (62,248)
  Fold 4: train -> 2021-08-02 (497,988) | test 2021-08-02 -> 2023-01-12 (62,248)
  Fold 5: train -> 2023-01-12 (560,236) | test 2023-01-12 -> 2024-06-28 (62,248)


## 4. LightGBM Config

In [56]:
def get_lgbm_params():
    return {
        'objective':         'binary',
        'metric':            'binary_logloss',
        'boosting_type':     'gbdt',
        'n_estimators':      5000,
        'learning_rate':     0.02,
        'num_leaves':        64,
        'max_depth':         6,
        'min_child_samples': 50,
        'feature_fraction':  0.7,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
        'device':            'gpu',
    }

print('Params ready. objective=binary, device=gpu.')


Params ready. objective=binary, device=gpu.


## 5. Train Both Models (Walk-Forward CV)

In [57]:
oof_probs_long  = np.full(len(X_clean), np.nan)
oof_probs_short = np.full(len(X_clean), np.nan)
best_iters = {'long': [], 'short': []}
cv_auc     = {'long': [], 'short': []}

splits = walk_forward_splits(len(X_clean))

for name, y_all in [('long', y_long_train), ('short', y_short_train)]:
    print(f'\nTraining {name} model...')
    for fold, (train_idx, test_idx) in enumerate(splits):
        X_tr, y_tr = X_clean.iloc[train_idx], y_all.iloc[train_idx]
        X_te, y_te = X_clean.iloc[test_idx],  y_all.iloc[test_idx]

        model = lgb.LGBMClassifier(**get_lgbm_params())
        model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

        probs = model.predict_proba(X_te)[:, 1]
        if name == 'long':  oof_probs_long[test_idx]  = probs
        else:               oof_probs_short[test_idx] = probs

        auc = roc_auc_score(y_te, probs)
        cv_auc[name].append(auc)
        best_iters[name].append(model.best_iteration_)
        print(f'  Fold {fold+1}: AUC={auc:.4f}  iters={model.best_iteration_}')
        del model, X_tr, y_tr, X_te, y_te; gc.collect()

    print(f'  CV mean AUC: {np.mean(cv_auc[name]):.4f} | avg iters: {int(np.mean(best_iters[name]))}')



Training long model...
  Fold 1: AUC=0.6985  iters=765
  Fold 2: AUC=0.7059  iters=574
  Fold 3: AUC=0.6940  iters=441
  Fold 4: AUC=0.7071  iters=834
  Fold 5: AUC=0.7269  iters=1043
  CV mean AUC: 0.7065 | avg iters: 731

Training short model...
  Fold 1: AUC=0.6952  iters=680
  Fold 2: AUC=0.7006  iters=632
  Fold 3: AUC=0.6874  iters=625
  Fold 4: AUC=0.6731  iters=643
  Fold 5: AUC=0.6929  iters=1134
  CV mean AUC: 0.6899 | avg iters: 742


## 6. Train Final Models & Save

In [ ]:
print('Training final models on full training set...')
for name, y_all in [('long', y_long_train), ('short', y_short_train)]:
    avg_iter = max(50, int(np.mean(best_iters[name])))
    model    = lgb.LGBMClassifier(**{**get_lgbm_params(), 'n_estimators': avg_iter})
    model.fit(X_clean, y_all)

    joblib.dump({
        'model':        model,
        'direction':    name,
        'feature_cols': feature_cols,
        'train_end':    TRAIN_END,
        'atr_mult':     ATR_MULT,
        'cv_auc':       np.mean(cv_auc[name]),
        'n_iters':      avg_iter,
    }, MODELS_DIR / f'model_4H_{name}.joblib')

    size_mb = (MODELS_DIR / f'model_4H_{name}.joblib').stat().st_size / 1024 / 1024
    print(f'  {name}: {avg_iter} iters, AUC={np.mean(cv_auc[name]):.4f}, saved ({size_mb:.1f} MB)')
    del model; gc.collect()

np.save(MODELS_DIR / 'y_long_train.npy',  y_long_train.values)
np.save(MODELS_DIR / 'y_short_train.npy', y_short_train.values)

del X_clean, y_long_train, y_short_train; gc.collect()
print(f'Models saved to {MODELS_DIR}')


Training final models on full training set...
  long: 731 iters, AUC=0.7065, saved (4.7 MB)
  short: 742 iters, AUC=0.6899, saved (4.8 MB)
Models saved to ..\backend\models_11


In [ ]:
y_long_arr  = np.load(MODELS_DIR / 'y_long_train.npy')
y_short_arr = np.load(MODELS_DIR / 'y_short_train.npy')

for name, oof_probs, y_arr in [
    ('long',  oof_probs_long,  y_long_arr),
    ('short', oof_probs_short, y_short_arr),
]:
    valid  = ~np.isnan(oof_probs)
    probs  = oof_probs[valid]
    y_true = y_arr[valid].astype(int)
    auc     = roc_auc_score(y_true, probs)
    ll      = log_loss(y_true, probs)
    base_ll = log_loss(y_true, np.full(len(y_true), y_true.mean()))
    print(f'{name.upper()} OOF: AUC={auc:.4f}  LogLoss={ll:.4f}  Baseline={base_ll:.4f}')
    print(f'  Prob range       N     Hit rate    Edge')
    print('  ' + '-'*42)
    for lo, hi in [(0.0,0.3),(0.3,0.4),(0.4,0.5),(0.5,0.6),(0.6,0.7),(0.7,1.0)]:
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() < 50: continue
        hit  = y_true[mask].mean()
        edge = hit - y_true.mean()
        print(f'  [{lo:.1f}-{hi:.1f})  {mask.sum():>8,}   {hit:>8.1%}   {edge:>+7.1%}')
    print()


ValueError: operands could not be broadcast together with shapes (622488,) (638893,) 